[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter16_making_training_stable/notebook.ipynb)

# Chapter 16: Making Training Stable

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Chapter 4 ended with a warning. A sigmoid passes back at most a quarter of the slope it receives, so after twenty layers almost nothing reaches the start of the network, and the early layers stop learning. That chapter showed the problem on paper. This one shows it in a real network and then fixes it.

By the early 2010s the problem had a name and a hunting party. Xavier Glorot and Yoshua Bengio published a careful study of it in 2010, and argued that the starting weights were part of the trouble. Then, in February 2015, two answers arrived five days apart. On 6 February a team at Microsoft Research in Beijing led by Kaiming He posted a rule for choosing starting weights for ReLU networks. On 11 February Sergey Ioffe and Christian Szegedy at Google posted a layer that rescales the numbers flowing through a network on every batch. They called it **batch normalization**, and reported reaching the same accuracy as a leading image model in fourteen times fewer training steps.

This chapter covers both, and then closes two threads. Chapter 6 listed four reasons that gradient descent works better than it ought to, and the last two of them, careful starting weights and a recipe that holds training together, arrive here. Chapter 14's table of symptoms ended on a loss stuck at the baseline, and that row is answered here too.

Before you read on, try to guess the answers to these four questions.

1. **If every layer shrinks the slope a little, what happens over twenty layers?**
2. **Could the random numbers a network starts with decide whether it can learn at all?**
3. **What would it mean to rescale the numbers inside a network on every batch?**
4. **When a network will not learn, what is the first thing to check?**

## Vanishing and Exploding Gradients

Use the field of plants from Chapter 1, which a small network learned perfectly. This time the network is twenty hidden layers deep.

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=400, noise=0.08,
                    factor=0.45, random_state=7)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y)
loss_fn = nn.CrossEntropyLoss()

The builder below stacks a chosen number of layers. Two switches are off for now: a way to set the starting weights by hand, and batch normalization.

In [ ]:
def deep(depth=20, width=32, act=nn.Sigmoid, init=None, bn=False, seed=0):
    torch.manual_seed(seed)
    layers, size = [], 2
    for i in range(depth):
        linear = nn.Linear(size, width)
        if init is not None:
            init(linear.weight)
            nn.init.zeros_(linear.bias)
        layers.append(linear)
        if bn:
            layers.append(nn.BatchNorm1d(width))
        layers.append(act())
        size = width
    layers.append(nn.Linear(size, 2))
    return nn.Sequential(*layers)

Now run one backward pass and measure how large the gradient is at each layer's weights.

In [ ]:
def gradient_sizes(net):
    net.zero_grad()
    loss_fn(net(X), y).backward()
    return [m.weight.grad.norm().item()
            for m in net if isinstance(m, nn.Linear)]

sizes = gradient_sizes(deep(act=nn.Sigmoid))
print("first layer %.1e" % sizes[0])
print("last layer  %.1e" % sizes[-1])

The first layer's gradient is about $10^{-19}$, and the last layer's is about $10^{-1}$. That is a ratio of roughly a billion billion. With any learning rate that suits the last layer, the first layer's weights move by an amount so small it is effectively zero, so the early layers are frozen at their random starting values for good.

This is Chapter 4's whisper line, measured. Figure 16.1 plots all twenty layers on a log scale, beside the same network built the way the next section recommends.

> 🖼️ **Figure 16.1:** How much gradient reaches each layer of a twenty layer network. With sigmoids it shrinks by about ten times per layer on the way back; with ReLU and a careful start it stays within a factor of about ten from end to end.

The opposite failure exists as well. If each layer multiplies the signal up instead of down, the numbers grow exponentially and you get **exploding gradients**, which end in the `inf` and `nan` of Chapter 14. Both failures come from the same place: a long chain of multiplications in which each factor is a little away from one. Raise 0.9 to the twentieth power and you get 0.12; raise 1.1 to the twentieth and you get 6.7; and real layers are usually much further from one than that.

## Weight Initialization

Switching the sigmoid for ReLU removes the hard limit of a quarter per layer, but it does not by itself make a deep network trainable. The size of the starting weights decides whether the signal grows, shrinks or holds steady as it passes through. To see that directly, measure the spread of the numbers coming out of each layer on the forward pass.

In [ ]:
def spreads(net):
    out, h = [], X
    with torch.no_grad():
        for m in net:
            h = m(h)
            if isinstance(m, nn.ReLU):
                out.append(h.std().item())
    return out

starts = {
    "tiny":    lambda w: nn.init.normal_(w, 0, 0.01),
    "default": None,
    "large":   lambda w: nn.init.normal_(w, 0, 1.0),
    "He":      lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
}
for name, init in starts.items():
    s = spreads(deep(act=nn.ReLU, init=init))
    print(name, ["%.2g" % s[i] for i in [0, 9, 19]])

The three columns are the spread after layers 1, 10 and 20. Weights drawn with a spread of 0.01 shrink the signal to nothing within a few layers. Weights with a spread of 1.0 blow it up past ten billion. PyTorch's default starting weights are sized for ordinary shallow networks and drift downward here. The **He initialization**, `nn.init.kaiming_normal_`, holds the spread roughly steady all the way through.

The idea behind it fits in one sentence. ReLU zeroes about half its inputs, which halves the spread of the signal, so the weights should be scaled to double it back, which works out to drawing them with a spread of $\sqrt{2/n}$, where $n$ is the number of inputs to the layer. For sigmoid and tanh networks the matching rule is Glorot's, `nn.init.xavier_normal_`.

> 🖼️ **Figure 16.2:** The spread of the signal as it passes through twenty ReLU layers, for four ways of choosing the starting weights. Only the He start keeps it level; the 0.01 start falls below the bottom of the chart after layer seven.

Now train all of them, plus the sigmoid network, for three hundred full batch steps with Adam.

In [ ]:
def train(net, steps=300, lr=0.01):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for step in range(steps):
        net.train()
        opt.zero_grad()
        loss = loss_fn(net(X), y)
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        accuracy = (net(X).argmax(1) == y).float().mean().item()
    return round(loss.item(), 4), round(accuracy, 3)

print("sigmoid", train(deep(act=nn.Sigmoid)))
for name, init in starts.items():
    print(name, train(deep(act=nn.ReLU, init=init)))

Look at the number 0.6931. That is $\ln 2$, the know nothing baseline from Chapter 5, and accuracy 0.5 is a coin toss. Two of these networks trained for three hundred steps and finished exactly where an untrained network starts, and the PyTorch default managed only 0.6919, a hair below it, with the same coin toss accuracy. Nothing crashed, nothing printed a warning, and the loss curve would show a flat line. That is the stuck at the baseline row from Chapter 14's table, and its cause here is not the learning rate at all. It is the starting weights.

The large start ended well only because Adam rescales each weight's step. Its first loss was over thirteen billion, and with plain SGD it does not survive a single step.

In [ ]:
net = deep(act=nn.ReLU, init=starts["large"])
opt = torch.optim.SGD(net.parameters(), lr=0.01)
for step in range(3):
    opt.zero_grad()
    loss = loss_fn(net(X), y)
    loss.backward()
    opt.step()
    print("%.4g" % loss.item())

That is the `nan` row of Chapter 11's table with a cause that no learning rate can fix. One choice of starting weights stops the network learning at all, another destroys it in one step, and a third, a single line of code, makes the same twenty layers learn the field perfectly. This is the third of Chapter 6's four reasons: where you start decides where, and whether, you arrive.

## Batch Normalization

Ioffe and Szegedy's idea attacks the same problem from inside the network. Rather than hoping the starting weights keep every layer's output at a sensible scale for the whole of training, measure each layer's output on every batch and rescale it on the spot.

> 🖼️ **Figure 16.3:** What one batch normalization layer does to each of its columns. The last step lets the network undo the rescaling if that turns out to help, so nothing is lost.

In a network the layer usually goes straight after a linear layer and before the activation, which is the order used in this chapter, so that the activation always receives numbers near zero where it does its most useful work. You will see the other order in published code as well, and both train well in practice.

Watch it work on a batch of three columns with wildly different scales.

In [ ]:
torch.manual_seed(0)
bn = nn.BatchNorm1d(3)
batch = torch.randn(64, 3) * torch.tensor([1., 10., 100.]) \
        + torch.tensor([5., -20., 300.])
out = bn(batch)
print([round(v, 1) for v in batch.mean(0).tolist()])
print([round(v, 1) for v in batch.std(0).tolist()])
print([round(v, 2) + 0.0 for v in out.mean(0).tolist()])
print([round(v, 2) for v in out.std(0).tolist()])

Means of 5, $-22.5$ and 326.8 and spreads from 0.9 to 105 all come out with mean 0 and spread 1. The layer after this one no longer has to cope with whatever scale the layer before it happened to drift to.

There is a train and eval distinction again, just as with dropout in Chapter 15. In training mode the layer uses the statistics of the current batch. At prediction time there may be only one example, which has no spread at all, so the layer uses running averages it has been collecting all along.

In [ ]:
print([round(v, 1) for v in bn.running_mean.tolist()])

Each running mean has moved one tenth of the way from 0 towards this batch's mean, because PyTorch updates it with a momentum of 0.1. Over hundreds of batches it settles on the average across the whole training set, and that is what `net.eval()` switches the layer to use. Forget `net.eval()` and a batch normalization network predicts differently depending on what else happens to be in the batch.

Now put it into the networks that would not train.

In [ ]:
print("sigmoid + bn", train(deep(act=nn.Sigmoid, bn=True)))
print("tiny + bn", train(deep(act=nn.ReLU, init=starts["tiny"], bn=True)))

Both networks that sat on the baseline now fit the field perfectly. Batch normalization rescued even the start whose signal had shrunk below $10^{-12}$, because it rescales that signal back up at every layer. That is what Ioffe and Szegedy meant when they wrote that it lets you be much less careful about initialization.

## Putting the Training Recipe Together

Chapters 12 to 16 have each added one ingredient. Table 16.1 gathers them into the default that most people start from, which is the fourth of Chapter 6's reasons, the recipe that holds everything else together.

**Table 16.1: A default training recipe. None of it is a law, and Chapters 13 to 15 each showed a case where a default lost, but it is a good place to start and a good place to return to when something breaks.**

| Ingredient | Sensible default | From |
|---|---|---|
| Inputs | Scaled to mean 0 and spread 1 | Chapter 14 |
| Batches | 16 to 256 examples, shuffled | Chapter 12 |
| Activation | ReLU | Chapter 4 |
| Starting weights | He for ReLU, Glorot for tanh | This chapter |
| Normalization | BatchNorm after each linear layer in a deep network | This chapter |
| Optimizer | Adam | Chapter 13 |
| Learning rate | From a range test, with a decay schedule | Chapter 14 |
| Regularization | Weight decay, dropout if needed, early stopping | Chapter 15 |

In code, a deep network built to this recipe looks like this.

In [ ]:
class SteadyNet(nn.Module):
    def __init__(self, inputs, width, depth, outputs):
        super().__init__()
        layers, size = [], inputs
        for i in range(depth):
            linear = nn.Linear(size, width)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.BatchNorm1d(width), nn.ReLU()]
            size = width
        layers.append(nn.Linear(size, outputs))
        self.body = nn.Sequential(*layers)

    def forward(self, x):
        return self.body(x)

print(train(SteadyNet(2, 32, 20, 2)))

## When Your Network Will Not Train: A Troubleshooting Guide

You now know every common cause, so here they are in the order worth checking. The single most useful test comes first: take a handful of training examples and see whether the network can fit them perfectly. A healthy network with working code can memorise eight examples in seconds. If it cannot, no amount of tuning on the full data will help, because the fault is in the model or the code, not the data.

In [ ]:
def fit_tiny(net, steps=200):
    xb, yb = X[:8], y[:8]
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for step in range(steps):
        opt.zero_grad()
        loss = loss_fn(net(xb), yb)
        loss.backward()
        opt.step()
    return round(loss.item(), 4)

print("healthy", fit_tiny(deep(act=nn.ReLU, init=starts["He"])))
print("stuck  ", fit_tiny(deep(act=nn.Sigmoid)))

The healthy network drives the loss to zero. The twenty layer sigmoid network cannot fit eight points, which tells you in a few seconds that the problem is structural before you spend an afternoon on learning rates.

**Table 16.2: Troubleshooting a network that will not train. The value $\ln k$ is the know nothing loss for $k$ classes, 0.693 for two and 1.099 for three.**

| Symptom | Check first | Then |
|---|---|---|
| Cannot fit eight examples | Shapes, labels, `zero_grad` order | Starting weights, depth |
| Loss stuck at $\ln k$ | Starting weights, saturated activations | Add BatchNorm, use He start |
| Loss is nan on step one | Starting weights, unscaled inputs | Then the learning rate |
| Loss is nan later | Learning rate | Decay schedule, gradient size |
| Loss falls very slowly | Learning rate too small | Range test |
| Validation rises as training falls | Overfitting | Chapter 15's three tools |
| Results differ every run | Seeds, a forgotten `net.eval()` | Average several seeds |

Read the second row's first entry with care. A loss sitting exactly on $\ln k$ is never a coincidence. It means the network is predicting equal probabilities for every class, which means no useful signal is reaching the output, and the cause is almost always upstream of the optimizer.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Find the depth where it breaks.** Train sigmoid networks of depth 2, 5, 10 and 20 with the default start and find the depth at which the loss stops leaving $\ln 2$.
2. **Try Glorot on sigmoids.** Pass `nn.init.xavier_normal_` as `init` to the twenty layer sigmoid network and see whether a careful start is enough without batch normalization.
3. **Catch the eval bug.** Train `SteadyNet`, then predict on a single example with and without `net.eval()`. One of them raises an error; explain why.
4. **Break the eight example test.** Introduce a bug on purpose, such as calling `opt.zero_grad()` after `loss.backward()`, and see whether `fit_tiny` catches it.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why does the gradient at the first layer of a deep sigmoid network become so small?**
2. **What does He initialization try to keep constant from layer to layer?**
3. **What does a loss of exactly 0.6931 on a two class problem tell you?**
4. **What four things does a batch normalization layer do to each column?**
5. **Why does batch normalization behave differently in training and evaluation mode?**
6. **Why is fitting eight examples the first test to run?**

### Answers

1. Because the backward pass multiplies by one slope per layer, and a sigmoid's slope is at most 0.25, so twenty layers multiply the signal by a tiny number.
2. The spread of each layer's output, so the signal neither shrinks to nothing nor grows without limit as it passes through.
3. That the network gives each class a probability of one half, which is the know nothing baseline. No useful signal is reaching the output.
4. It subtracts the batch mean, divides by the batch spread, then multiplies by a learned scale and adds a learned shift.
5. Because a single example at prediction time has no spread to measure, so the layer uses running averages collected during training instead of the current batch's statistics.
6. Because a healthy network can always memorise a handful of examples, so failing tells you quickly that the fault is in the model or the code rather than in the data or the tuning.

## What Comes Next

You can now measure vanishing and exploding signals, choose starting weights that keep a deep network alive, use batch normalization with its train and eval behaviour in mind, and work through a failing run in a sensible order. That completes Part 4.

- **Chapter 17, Convolutional Neural Networks,** meets data that the dense layers of every chapter so far handle badly: images.
- **Chapter 18, Sequence Models and Embeddings,** meets data that comes in order, and symbols that are not numbers at all.

Chapter 17 begins with a question that sounds too simple: why should a network looking for a cat in the corner of a photograph have to learn the cat separately for every corner?